# Mean Group Estimator - Exercise Solutions

Solutions for exercises in `advanced/01_mean_group_estimator.ipynb`.

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import panelbox as pb

np.random.seed(42)
plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['font.size'] = 11

data = pb.load_grunfeld()

---
## Exercise 1: Weighted Mean Group

Fit a weighted MG using the inverse of the entity-specific variance of `value` as weights.

In [ ]:
# First, fit unweighted MG to get entity-specific standard errors
mg_unw = pb.MeanGroupEstimator('invest ~ value + capital', data, 'firm', 'year', min_obs_per_entity=5)
mg_unw_res = mg_unw.fit()

# Get entity-specific std errors for 'value'
entity_se = {eid: se['value'] for eid, se in mg_unw_res.entity_std_errors.items()}
print("Entity-specific std errors for 'value':")
for eid, se in sorted(entity_se.items()):
    print(f"  Firm {eid}: {se:.6f}")

# Weights = inverse variance (1/se^2)
entities_used = list(mg_unw_res.entity_params.keys())
weights = np.array([1.0 / entity_se[e]**2 for e in entities_used])
weights = weights / weights.sum() * len(weights)  # normalize to sum = N

print(f"\nNormalized weights: {weights.round(3)}")

In [ ]:
# Fit weighted MG
mg_w = pb.MeanGroupEstimator(
    'invest ~ value + capital', data, 'firm', 'year',
    weights=weights, min_obs_per_entity=5
)
mg_w_res = mg_w.fit()

# Compare
comparison = pd.DataFrame({
    'Unweighted MG': mg_unw_res.params,
    'Weighted MG': mg_w_res.params,
}).round(4)

print("Comparison: Unweighted vs Weighted MG")
print("=" * 50)
print(comparison.to_string())
print()
print("The weighted MG gives more weight to firms with more")
print("precisely estimated 'value' coefficients.")

---
## Exercise 2: MG with Simulated Data (Mild Heterogeneity)

In [ ]:
# Simulate
N, T = 30, 50
beta_mean, beta_std = 3.0, 0.5  # mild heterogeneity

rows = []
betas_true = {}
for i in range(1, N + 1):
    beta_i = np.random.normal(beta_mean, beta_std)
    alpha_i = np.random.normal(0, 2)
    betas_true[i] = beta_i
    for t in range(1, T + 1):
        x = np.random.normal(0, 2)
        y = alpha_i + beta_i * x + np.random.normal(0, 1)
        rows.append({'entity': i, 'time': t, 'y': y, 'x': x})

sim_data = pd.DataFrame(rows)

true_avg = np.mean(list(betas_true.values()))
print(f"True average beta: {true_avg:.4f}")
print(f"True beta std: {np.std(list(betas_true.values()), ddof=1):.4f}")

In [ ]:
# FE vs MG
fe_sim = pb.FixedEffects('y ~ x', sim_data, 'entity', 'time')
fe_sim_res = fe_sim.fit()

mg_sim = pb.MeanGroupEstimator('y ~ x', sim_data, 'entity', 'time', min_obs_per_entity=5)
mg_sim_res = mg_sim.fit()

print(f"True average slope:  {true_avg:.4f}")
print(f"FE estimate:         {fe_sim_res.params['x']:.4f}")
print(f"MG estimate:         {mg_sim_res.params['x']:.4f}")
print()

swamy = mg_sim_res.swamy_test_result
print(f"Swamy test statistic: {swamy['statistic']:.4f}")
print(f"Swamy p-value:        {swamy['pvalue']:.4f}")
print(f"Reject H0 (5%):      {swamy['reject']}")
print()
print("With mild heterogeneity (std=0.5), the Swamy test may or")
print("may not reject. Both FE and MG should be close to the true")
print("average since x is exogenous (not correlated with alpha_i).")

---
## Exercise 3: PMG with Different Lags

In [ ]:
# PMG with lags=1
pmg1 = pb.PooledMeanGroupEstimator('invest ~ value + capital', data, 'firm', 'year', lags=1)
pmg1_res = pmg1.fit()

# PMG with lags=2
pmg2 = pb.PooledMeanGroupEstimator('invest ~ value + capital', data, 'firm', 'year', lags=2)
pmg2_res = pmg2.fit()

comparison = pd.DataFrame({
    'PMG (lags=1)': pmg1_res.long_run_params,
    'PMG (lags=2)': pmg2_res.long_run_params,
}).round(4)

print("Long-Run Coefficients: PMG with Different Lag Orders")
print("=" * 55)
print(comparison.to_string())
print()
print(f"PMG(1) converged: {pmg1_res.converged} (iterations: {pmg1_res.n_iter})")
print(f"PMG(2) converged: {pmg2_res.converged} (iterations: {pmg2_res.n_iter})")
print()
print("The long-run coefficients may differ somewhat between lag")
print("specifications. With short T (=20), higher-order lags consume")
print("more degrees of freedom and estimates may be less stable.")